# 03 — Dynamic Gesture Recognition — Temporal Transformer Training

> **GPU Required** — CUDA hardware with ≥ 8 GB VRAM recommended.

**Pipeline:**
1. Load temporal landmark sequences (T × D)
2. Define Temporal Transformer architecture
3. Train with: warmup → cosine annealing · gradient clipping · EMA · early stopping · mixed precision
4. Progressive layer unfreezing schedule
5. Evaluate on validation & test sets (Top-1 / Top-5)
6. Export to ONNX (large + small variants)


In [ ]:
import os, json, math, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import onnx

ROOT           = Path("..")
DATA_DYN       = ROOT / "data" / "landmarks" / "dynamic"
MODELS_DIR     = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
CFG = dict(
    seq_len        = 30,       # frames per gesture window
    input_dim      = 126,      # 2 hands × 21 × 3
    d_model        = 256,      # transformer hidden dim
    nhead          = 8,        # attention heads
    num_layers     = 6,        # transformer encoder layers
    dim_feedforward= 512,
    dropout        = 0.1,
    label_smoothing= 0.1,
    lr_peak        = 1e-4,
    lr_min         = 1e-6,
    weight_decay   = 1e-4,
    batch_size     = 64,
    epochs         = 150,
    warmup_epochs  = 10,
    patience       = 20,
    grad_clip      = 1.0,
    ema_decay      = 0.9995,
    # Progressive unfreezing: unfreeze one layer every N epochs
    unfreeze_every = 15,
)
print(CFG)

In [ ]:
# ── Load data ────────────────────────────────────────────────
def load_split(split):
    p = DATA_DYN / split
    X = torch.tensor(np.load(p / "X.npy"), dtype=torch.float32)  # (N, T, D)
    y = torch.tensor(np.load(p / "y.npy"), dtype=torch.long)
    return X, y

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")

NUM_CLASSES = int(y_train.max().item()) + 1
print(f"Train  : {X_train.shape}")
print(f"Val    : {X_val.shape}")
print(f"Test   : {X_test.shape}")
print(f"Classes: {NUM_CLASSES}")

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=CFG["batch_size"], shuffle=True, drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),
                          batch_size=CFG["batch_size"], shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),
                          batch_size=CFG["batch_size"], shuffle=False)

In [ ]:
# ── Positional Encoding ──────────────────────────────────────
class SinusoidalPE(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):   # x: (B, T, D)
        return self.drop(x + self.pe[:, :x.size(1)])


# ── Temporal Transformer ─────────────────────────────────────
class TemporalTransformer(nn.Module):
    """
    Transformer encoder for gesture sequence classification.
    Learns motion, speed, trajectory, and temporal context.
    """
    def __init__(self, input_dim, d_model, nhead, num_layers,
                 dim_feedforward, dropout, num_classes):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc    = SinusoidalPE(d_model, dropout=dropout)

        enc_layer  = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                             enable_nested_tensor=False)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x):  # x: (B, T, D)
        B = x.size(0)
        x = self.input_proj(x)                          # (B, T, d_model)
        cls = self.cls_token.expand(B, -1, -1)          # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)                # (B, T+1, d_model)
        x   = self.pos_enc(x)
        x   = self.encoder(x)                           # (B, T+1, d_model)
        return self.head(x[:, 0])                       # CLS token → logits


model = TemporalTransformer(
    input_dim=CFG["input_dim"],
    d_model=CFG["d_model"],
    nhead=CFG["nhead"],
    num_layers=CFG["num_layers"],
    dim_feedforward=CFG["dim_feedforward"],
    dropout=CFG["dropout"],
    num_classes=NUM_CLASSES,
).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

In [ ]:
# ── Progressive layer unfreezing ─────────────────────────────
def freeze_encoder_layers(model, n_frozen: int):
    """Freeze first n_frozen transformer encoder layers."""
    for i, layer in enumerate(model.encoder.layers):
        for p in layer.parameters():
            p.requires_grad = (i >= n_frozen)

# Start with all encoder layers frozen (only head trains)
n_total_layers = CFG["num_layers"]
freeze_encoder_layers(model, n_total_layers)
print(f"Initial: {n_total_layers} encoder layers frozen")

In [ ]:
# ── EMA ─────────────────────────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}
    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)
    def apply_shadow(self, model):
        model.load_state_dict(self.shadow)

# ── Warmup + Cosine scheduler ────────────────────────────────
class WarmupCosineScheduler(optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs, lr_min=1e-6, last_epoch=-1):
        self.warmup  = warmup_epochs
        self.total   = total_epochs
        self.lr_min  = lr_min
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch
        if ep < self.warmup:
            factor = (ep + 1) / max(self.warmup, 1)
        else:
            progress = (ep - self.warmup) / max(self.total - self.warmup, 1)
            factor   = self.lr_min / self.base_lrs[0] + \
                       0.5 * (1 - self.lr_min / self.base_lrs[0]) * (1 + math.cos(math.pi * progress))
        return [base_lr * factor for base_lr in self.base_lrs]

ema       = EMA(model, decay=CFG["ema_decay"])
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=CFG["lr_peak"], weight_decay=CFG["weight_decay"])
scheduler = WarmupCosineScheduler(optimizer,
                                  CFG["warmup_epochs"], CFG["epochs"],
                                  lr_min=CFG["lr_min"])
scaler    = GradScaler()
writer    = SummaryWriter(log_dir=str(ROOT / "logs" / "dynamic"))

In [ ]:
# ── Top-k accuracy ────────────────────────────────────────────
def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k=5):
    topk = logits.topk(min(k, logits.size(-1)), dim=-1).indices
    correct = topk.eq(targets.view(-1, 1).expand_as(topk))
    return correct.any(dim=-1).float().mean().item()

In [ ]:
# ── Training loop ─────────────────────────────────────────────
best_val_acc  = 0.0
best_ckpt     = None
patience_ctr  = 0
n_frozen      = n_total_layers
history       = {"train_loss": [], "val_acc": [], "val_top5": [], "lr": []}

for epoch in range(1, CFG["epochs"] + 1):

    # ── Progressive unfreezing ───────────────────────────────
    if epoch > 1 and (epoch - 1) % CFG["unfreeze_every"] == 0 and n_frozen > 0:
        n_frozen -= 1
        freeze_encoder_layers(model, n_frozen)
        # Re-create optimizer with updated params
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CFG["lr_peak"] * (0.5 ** (n_total_layers - n_frozen)),
            weight_decay=CFG["weight_decay"],
        )
        print(f"  [Unfreeze] layers frozen: {n_frozen}")

    # ── Train ─────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    cur_lr   = scheduler.get_last_lr()[0]

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            logits = model(X_b.to(DEVICE)).cpu()
            all_logits.append(logits); all_labels.append(y_b)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    val_acc  = accuracy_score(all_labels, all_logits.argmax(1))
    val_top5 = topk_accuracy(all_logits, all_labels, k=5)

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["val_top5"].append(val_top5)
    history["lr"].append(cur_lr)

    writer.add_scalar("Loss/train",      avg_loss, epoch)
    writer.add_scalar("Accuracy/val",    val_acc,  epoch)
    writer.add_scalar("Top5/val",        val_top5, epoch)
    writer.add_scalar("LR",              cur_lr,   epoch)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_ckpt    = deepcopy(model.state_dict())
        patience_ctr = 0
        print(f"  ★ Epoch {epoch:3d} | loss {avg_loss:.4f} | acc {val_acc:.4f} | top5 {val_top5:.4f} | lr {cur_lr:.2e}")
    else:
        patience_ctr += 1
        if epoch % 10 == 0:
            print(f"    Epoch {epoch:3d} | loss {avg_loss:.4f} | acc {val_acc:.4f} | top5 {val_top5:.4f}")

    if patience_ctr >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch}")
        break

writer.close()
print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
# ── Test evaluation ───────────────────────────────────────────
model.load_state_dict(best_ckpt)
ema.apply_shadow(model)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        all_preds.append(model(X_b.to(DEVICE)).argmax(1).cpu())
        all_labels.append(y_b)

preds  = torch.cat(all_preds).numpy()
truths = torch.cat(all_labels).numpy()
print(classification_report(truths, preds, zero_division=0))

In [ ]:
# ── Training curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], color="steelblue")
axes[0].set_title("Training Loss")
axes[1].plot(history["val_acc"],    color="darkorange", label="Top-1")
axes[1].plot(history["val_top5"],   color="green",      label="Top-5")
axes[1].legend(); axes[1].set_title("Validation Accuracy")
axes[2].plot(history["lr"], color="purple")
axes[2].set_title("Learning Rate")
for ax in axes: ax.set_xlabel("Epoch")
plt.tight_layout(); plt.show()

In [ ]:
# ── ONNX Export — Large ───────────────────────────────────────
model.eval().to("cpu")
dummy = torch.zeros(1, CFG["seq_len"], CFG["input_dim"])
onnx_large = str(MODELS_DIR / "arabic_sign_video_model_large.onnx")

torch.onnx.export(
    model, dummy, onnx_large,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["sequence"],
    output_names=["logits"],
    dynamic_axes={"sequence": {0: "batch"}, "logits": {0: "batch"}},
)
onnx.checker.check_model(onnx.load(onnx_large))
print(f"Large ONNX → {onnx_large}")

In [ ]:
# ── Small model variant (fewer layers / smaller d_model) ──────
small_cfg = {**CFG, "d_model": 64, "nhead": 4, "num_layers": 2, "dim_feedforward": 128}

small_model = TemporalTransformer(
    input_dim=small_cfg["input_dim"],
    d_model=small_cfg["d_model"],
    nhead=small_cfg["nhead"],
    num_layers=small_cfg["num_layers"],
    dim_feedforward=small_cfg["dim_feedforward"],
    dropout=small_cfg["dropout"],
    num_classes=NUM_CLASSES,
)
# (Optionally fine-tune small_model via knowledge distillation from large model)

onnx_small = str(MODELS_DIR / "arabic_sign_video_model_small.onnx")
torch.onnx.export(
    small_model, dummy, onnx_small,
    export_params=True, opset_version=17, do_constant_folding=True,
    input_names=["sequence"], output_names=["logits"],
    dynamic_axes={"sequence": {0: "batch"}, "logits": {0: "batch"}},
)
onnx.checker.check_model(onnx.load(onnx_small))
print(f"Small ONNX → {onnx_small}")
print(f"Large params : {sum(p.numel() for p in model.parameters()):,}")
print(f"Small params : {sum(p.numel() for p in small_model.parameters()):,}")